In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import display
except Exception:
    display = None


# ============================================================
# Metric configuration
# ============================================================
# Current orientation:
#   x-axis = control-pseudo local mixing score
#   y-axis = perturbation-effect Pearson
# If you want to swap orientation again, only change X_COLUMN/Y_COLUMN
# and X_LABEL/Y_LABEL below; the mean reference lines and limits update automatically.

X_COLUMN = "control_manifold__control_pseudo_local_mixing_score_mean"
Y_COLUMN = "perturbation_effect__perturbation_effect_pearson_mean"

X_LABEL = "Control-pseudo local mixing score"
Y_LABEL = "Perturbation-effect Pearson"
PLOT_TITLE = "Control manifold preservation vs perturbation-effect recovery"
SAVE_NAME = "control_mixing_vs_perturb_effect_pearson_scatter_mean_lines_keep_s0"


# ============================================================
# Dataset/group discovery
# ============================================================

DATASET_NAMES = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

GROUPS = ["single", "dual", "multi"]

SELECT_CSV_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"
OUTPUT_DIR = "Perturbation/evaluation/scatter_plots"


# ============================================================
# Axis limit controls
# ============================================================
# X_LIM/Y_LIM:
#   None -> fully automatic.
#   (lower, upper) -> manual bounds.
#   (None, upper) or (lower, None) -> one bound automatic.
#
# USE_GLOBAL_AUTO_LIMITS=True computes one common x/y limit across all discovered
# dataset/group panels, using only selected non-S0 strategies. This makes panels
# directly comparable across datasets.

X_LIM = None  # type: Optional[Tuple[Optional[float], Optional[float]]]
Y_LIM = None  # type: Optional[Tuple[Optional[float], Optional[float]]]

X_LIM_BY_DATASET_GROUP = {}  # type: Dict[Tuple[str, str], Tuple[Optional[float], Optional[float]]]
Y_LIM_BY_DATASET_GROUP = {
    ("Replogle_K562_essential", "single"): (0.70, 1.01),
    ("Replogle_RPE", "single"): (0.85, 1.01),
    ("NormanWeissman2019", "single"): (0.90, 1.01),
    ("NormanWeissman2019", "dual"): (0.95, 1.01),
    ("ChangYe", "single"): (0.99, 1.01),
    ("ZhaoSims2021", "single"): (0.95, 1.01),
}  # type: Dict[Tuple[str, str], Tuple[Optional[float], Optional[float]]]

USE_GLOBAL_AUTO_LIMITS = True

# X is local mixing here, so it is usually non-negative.
AUTO_X_LOWER_AT_ZERO = True

# Y is Pearson here, so do not force the axis to start at zero.
AUTO_Y_LOWER_AT_ZERO = False

AXIS_MARGIN_FRACTION = 0.08
MIN_AXIS_SPAN_X = 1e-3
MIN_AXIS_SPAN_Y = 1e-3


# ============================================================
# Mean reference lines
# ============================================================
# These lines are computed from all selected strategies included in the figure.
# S0 naive mean control is excluded.

SHOW_MEAN_LINES = True
MEAN_LINE_COLOR = "#333333"
MEAN_LINESTYLE = "--"
MEAN_LINEWIDTH = 1.35
MEAN_LINE_ALPHA = 0.70
SHOW_MEAN_LINE_LABELS_ON_PLOT = True
MEAN_LABEL_FONT_SIZE = 10.5


# ============================================================
# S0 naive-control point
# ============================================================
# Keep S0 as a visible point, but do not draw S0 reference lines.
# The mean lines above are computed only from the selected non-S0 strategies.

SHOW_S0_POINT = True
S0_POINT_COLOR = "#4D4D4D"
S0_POINT_MARKER = "X"
S0_POINT_SIZE = 135
S0_POINT_ALPHA = 0.92


# ============================================================
# Plot style
# ============================================================

matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["xtick.major.width"] = 0.8
matplotlib.rcParams["ytick.major.width"] = 0.8

FIGSIZE = (10, 10)
POINT_SIZE = 175
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.8

SHOW_GRID = False
SAVE_PNG = True
SAVE_SVG = True
DISPLAY_FIGURES = True

LEGEND_NCOL = 2
LEGEND_FONT_SIZE = 10
LEGEND_TITLE_FONT_SIZE = 10
LEGEND_Y_ANCHOR = -0.24
BOTTOM_MARGIN = 0.35

DEOVERLAP_POINTS = True
MIN_POINT_DISTANCE_FRACTION = 0.018
POINT_REPEL_ITERATIONS = 12
SHOW_DISPLACEMENT_CONNECTORS = True


# ============================================================
# Strategy labels and colors
# ============================================================

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#D0E0EF",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#9F8DB8",
}

S5_VARIANT_COLORS = {
    "200&5": "#D49AB5",
    # "350&5": "#7D5284",
    "350&5": "#9F8DB8", 
    "500&5": "#B66699",
}


# ============================================================
# Helper functions
# ============================================================

def as_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def is_valid_color(x: Any) -> bool:
    if pd.isna(x):
        return False
    x = str(x).strip()
    return bool(x) and x.lower() not in {"nan", "none", "null"}


def clean_label_for_legend(label: str) -> str:
    return " ".join(str(label).replace("\n", " ").split())


def extract_variant_suffix(display_label: str) -> str:
    display_label = str(display_label)
    if "(" in display_label and ")" in display_label:
        return display_label[display_label.rfind("("):].strip()
    return ""


def build_plot_label(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))
    base = STRATEGY_PLOT_LABELS.get(strategy, strategy)
    display_label = str(row.get("display_variant_label", ""))
    suffix = extract_variant_suffix(display_label)
    if suffix:
        return f"{base}\n{suffix}"
    return base


def choose_color(row: pd.Series) -> str:
    for col in ["manual_color", "color"]:
        if col in row.index and is_valid_color(row[col]):
            return str(row[col]).strip()

    strategy = str(row.get("strategy", ""))
    display_label = str(row.get("display_variant_label", ""))

    if strategy == "S5_SEACell_OT_sampled_average":
        for key, color in S5_VARIANT_COLORS.items():
            if key in display_label:
                return color

    return STRATEGY_BASE_COLORS.get(strategy, "#999999")


def get_selected_plot_df(df_all: pd.DataFrame) -> pd.DataFrame:
    """Return selected final strategies, excluding S0 naive mean control."""
    if "select_for_final" in df_all.columns:
        out = df_all[as_bool_series(df_all["select_for_final"])].copy()
    else:
        print("[Info] 'select_for_final' was not found; using all non-S0 rows.")
        out = df_all.copy()

    out = out[out["strategy"].astype(str) != "S0_naive_mean_control_reference"].copy()
    return out


def get_s0_reference_row(df_all: pd.DataFrame) -> Optional[pd.Series]:
    """Return the S0 naive mean control row if present and numeric for the plotted metrics."""
    ref = df_all[df_all["strategy"].astype(str) == "S0_naive_mean_control_reference"].copy()
    if ref.empty:
        return None
    ref[X_COLUMN] = pd.to_numeric(ref[X_COLUMN], errors="coerce")
    ref[Y_COLUMN] = pd.to_numeric(ref[Y_COLUMN], errors="coerce")
    ref = ref.dropna(subset=[X_COLUMN, Y_COLUMN]).copy()
    if ref.empty:
        return None
    ref["plot_color"] = ref.apply(choose_color, axis=1)
    ref["plot_label"] = ref.apply(build_plot_label, axis=1)
    return ref.iloc[0]


def prepare_plot_df(df_all: pd.DataFrame) -> pd.DataFrame:
    required_cols = ["strategy", X_COLUMN, Y_COLUMN]
    missing_cols = [c for c in required_cols if c not in df_all.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    df_plot = get_selected_plot_df(df_all)
    df_plot[X_COLUMN] = pd.to_numeric(df_plot[X_COLUMN], errors="coerce")
    df_plot[Y_COLUMN] = pd.to_numeric(df_plot[Y_COLUMN], errors="coerce")
    df_plot = df_plot.dropna(subset=[X_COLUMN, Y_COLUMN]).copy().reset_index(drop=True)

    if df_plot.empty:
        raise ValueError("No selected non-S0 rows available for the scatter plot.")

    df_plot["plot_color"] = df_plot.apply(choose_color, axis=1)
    df_plot["plot_label"] = df_plot.apply(build_plot_label, axis=1)
    return df_plot


def get_dataset_group_from_sub_path(sub_path: Path) -> Tuple[str, str]:
    dataset = sub_path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = sub_path.name
    return dataset, group


def get_dataset_group_title(sub_path: Path) -> str:
    dataset, group = get_dataset_group_from_sub_path(sub_path)
    return f"{dataset} | {group}"


def get_user_axis_limit(base_limit, per_group_limits, sub_path: Path):
    dataset, group = get_dataset_group_from_sub_path(sub_path)
    return per_group_limits.get((dataset, group), base_limit)


def choose_axis_limit(
    values: np.ndarray,
    user_limit,
    margin_fraction: float = AXIS_MARGIN_FRACTION,
    auto_lower_at_zero: bool = False,
    min_span: float = 1e-3,
) -> Tuple[float, float]:
    finite_values = np.asarray(values, dtype=float)
    finite_values = finite_values[np.isfinite(finite_values)]

    if finite_values.size == 0:
        raise ValueError("Cannot determine axis limit because all values are non-finite.")

    data_min = float(np.min(finite_values))
    data_max = float(np.max(finite_values))
    span = max(data_max - data_min, float(min_span))

    center = 0.5 * (data_min + data_max)
    if data_max - data_min < min_span:
        data_min = center - 0.5 * min_span
        data_max = center + 0.5 * min_span
        span = min_span

    if auto_lower_at_zero:
        auto_lower = 0.0
    else:
        auto_lower = data_min - margin_fraction * span

    auto_upper = data_max + margin_fraction * span

    if auto_upper <= auto_lower:
        auto_upper = auto_lower + float(min_span)

    if user_limit is None:
        return float(auto_lower), float(auto_upper)

    if len(user_limit) != 2:
        raise ValueError("Axis limit must be None or a 2-tuple, e.g. (0.0, 1.0).")

    lower, upper = user_limit
    lower = auto_lower if lower is None else float(lower)
    upper = auto_upper if upper is None else float(upper)

    if not np.isfinite(lower) or not np.isfinite(upper):
        raise ValueError(f"Axis limit contains non-finite value: {user_limit}")
    if upper <= lower:
        raise ValueError(f"Axis upper limit must be greater than lower limit: {user_limit}")

    return lower, upper


def deoverlap_xy(
    x_values: np.ndarray,
    y_values: np.ndarray,
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
    min_dist_fraction: float,
    n_iter: int,
    padding_fraction: float = 0.010,
) -> Tuple[np.ndarray, np.ndarray]:
    """De-overlap close points by moving them slightly in both x and y."""
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)

    n = len(x_values)
    if n <= 1:
        return x_values.copy(), y_values.copy()

    x_span = max(float(x_max - x_min), 1e-12)
    y_span = max(float(y_max - y_min), 1e-12)

    x_norm = (x_values - x_min) / x_span
    y_norm = (y_values - y_min) / y_span

    x_adj = x_norm.copy()
    y_adj = y_norm.copy()

    min_dist = float(min_dist_fraction)
    pad = float(padding_fraction)

    for _ in range(int(n_iter)):
        max_push = 0.0
        for i in range(n - 1):
            for j in range(i + 1, n):
                dx = float(x_adj[j] - x_adj[i])
                dy = float(y_adj[j] - y_adj[i])
                dist = float(np.sqrt(dx * dx + dy * dy))

                if dist >= min_dist:
                    continue

                if dist < 1e-12:
                    angle = ((i + j + 1) * np.pi / 5.0)
                    ux, uy = np.cos(angle), np.sin(angle)
                else:
                    ux, uy = dx / dist, dy / dist

                push_total = min_dist - dist
                push_each = 0.5 * push_total
                x_adj[i] -= ux * push_each
                y_adj[i] -= uy * push_each
                x_adj[j] += ux * push_each
                y_adj[j] += uy * push_each
                max_push = max(max_push, push_each)

        x_adj = np.clip(x_adj, pad, 1.0 - pad)
        y_adj = np.clip(y_adj, pad, 1.0 - pad)

        if max_push < 1e-5:
            break

    x_out = x_min + x_adj * x_span
    y_out = y_min + y_adj * y_span
    return x_out, y_out


def build_strategy_legend_label(row: pd.Series) -> str:
    label = clean_label_for_legend(row.get("plot_label", build_plot_label(row)))
    x_val = row.get(X_COLUMN, np.nan)
    y_val = row.get(Y_COLUMN, np.nan)
    if pd.notna(x_val) and pd.notna(y_val):
        label += f"  |  x={float(x_val):.3f}, y={float(y_val):.3f}"
    return label


def make_bottom_legend(
    ax,
    df_plot: pd.DataFrame,
    mean_x: float,
    mean_y: float,
    df_s0: Optional[pd.Series] = None,
):
    handles = []
    labels = []

    for _, row in df_plot.iterrows():
        handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="None",
                markersize=8.5,
                markerfacecolor=row["plot_color"],
                markeredgecolor=POINT_EDGE_COLOR,
                markeredgewidth=POINT_EDGE_WIDTH,
            )
        )
        labels.append(build_strategy_legend_label(row))

    if SHOW_S0_POINT and df_s0 is not None:
        s0_x = float(df_s0[X_COLUMN])
        s0_y = float(df_s0[Y_COLUMN])
        handles.append(
            Line2D(
                [0],
                [0],
                marker=S0_POINT_MARKER,
                linestyle="None",
                markersize=9.5,
                markerfacecolor=S0_POINT_COLOR,
                markeredgecolor=POINT_EDGE_COLOR,
                markeredgewidth=POINT_EDGE_WIDTH,
            )
        )
        labels.append(f"S0 naive mean control point  |  x={s0_x:.3f}, y={s0_y:.3f}")

    if SHOW_MEAN_LINES:
        handles.extend(
            [
                Line2D([0], [0], color=MEAN_LINE_COLOR, linestyle=MEAN_LINESTYLE, linewidth=MEAN_LINEWIDTH),
                Line2D([0], [0], color=MEAN_LINE_COLOR, linestyle=MEAN_LINESTYLE, linewidth=MEAN_LINEWIDTH),
            ]
        )
        labels.extend([
            f"Mean {X_LABEL}  |  x={mean_x:.3f}",
            f"Mean {Y_LABEL}  |  y={mean_y:.3f}",
        ])

    return ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, LEGEND_Y_ANCHOR),
        ncol=LEGEND_NCOL,
        frameon=False,
        fontsize=LEGEND_FONT_SIZE,
        title="Selected strategies, S0 point, and within-panel means",
        title_fontsize=LEGEND_TITLE_FONT_SIZE,
        handlelength=1.5,
        columnspacing=1.4,
        handletextpad=0.6,
        borderaxespad=0.0,
    )


def has_required_columns(csv_path: Path) -> bool:
    try:
        cols = pd.read_csv(csv_path, nrows=0).columns
    except Exception:
        return False
    required = {"strategy", X_COLUMN, Y_COLUMN}
    return required.issubset(set(cols))


def resolve_selected_csv(result_analysis_dir: Path, select_csv_name: str) -> Optional[Path]:
    direct = result_analysis_dir / select_csv_name
    if direct.exists():
        return direct

    candidates = sorted(result_analysis_dir.glob("*.csv"))
    valid = [p for p in candidates if has_required_columns(p)]

    if not valid:
        return None

    print(
        "[Info] Requested SELECT_CSV_NAME was not found. "
        f"Using detected metric table instead: {valid[0].name}"
    )
    return valid[0]


def discover_valid_paths(root_dir: Path) -> List[Path]:
    candidate_paths = [root_dir / f"{name}_pseudo_pairing_evaluation" for name in DATASET_NAMES]
    detailed_sub_paths = [sub_path / group for sub_path in candidate_paths for group in GROUPS]
    return [sub_path for sub_path in detailed_sub_paths if sub_path.exists()]


def relative_save_dir(root_dir: Path, sub_path: Path) -> Path:
    try:
        rel = sub_path.relative_to(root_dir)
    except ValueError:
        rel = Path(sub_path.parent.name) / sub_path.name
    return Path(OUTPUT_DIR) / rel


def collect_plot_inputs(root_dir: Path) -> List[dict[str, Any]]:
    """Read all valid selected tables and prepare selected non-S0 plot rows."""
    valid_paths = discover_valid_paths(root_dir)
    if not valid_paths:
        raise FileNotFoundError(
            f"No dataset/group folders found under {root_dir}. "
            "Check ROOT_DIR, DATASET_NAMES, and GROUPS."
        )

    items: List[dict[str, Any]] = []
    for sub_path in valid_paths:
        result_analysis_dir = sub_path / "result_analysis"
        select_csv = resolve_selected_csv(result_analysis_dir, SELECT_CSV_NAME)
        if select_csv is None:
            print(f"[Skip] No CSV with required columns found under: {result_analysis_dir}")
            continue
        print(f"[Read] {select_csv}")
        df_all = pd.read_csv(select_csv)
        try:
            df_plot = prepare_plot_df(df_all)
            df_s0 = get_s0_reference_row(df_all)
        except Exception as exc:
            print(f"[Skip] {sub_path}: {repr(exc)}")
            continue
        if SHOW_S0_POINT and df_s0 is None:
            print(f"[Warn] {sub_path}: S0 naive mean control row not found or invalid; S0 point will be omitted for this panel.")
        items.append({"sub_path": sub_path, "select_csv": select_csv, "df_plot": df_plot, "df_s0": df_s0})
    return items


def compute_global_axis_limits(plot_items: List[dict[str, Any]]) -> Tuple[Tuple[float, float], Tuple[float, float]]:
    x_values = []
    y_values = []
    for item in plot_items:
        df_plot = item["df_plot"]
        x_values.extend(pd.to_numeric(df_plot[X_COLUMN], errors="coerce").dropna().astype(float).tolist())
        y_values.extend(pd.to_numeric(df_plot[Y_COLUMN], errors="coerce").dropna().astype(float).tolist())

        # Include S0 in axis limits so the retained S0 point is never cropped.
        df_s0 = item.get("df_s0", None)
        if SHOW_S0_POINT and df_s0 is not None:
            s0_x = pd.to_numeric(pd.Series([df_s0[X_COLUMN]]), errors="coerce").iloc[0]
            s0_y = pd.to_numeric(pd.Series([df_s0[Y_COLUMN]]), errors="coerce").iloc[0]
            if pd.notna(s0_x):
                x_values.append(float(s0_x))
            if pd.notna(s0_y):
                y_values.append(float(s0_y))

    x_lim = choose_axis_limit(
        values=np.asarray(x_values, dtype=float),
        user_limit=X_LIM,
        auto_lower_at_zero=AUTO_X_LOWER_AT_ZERO,
        min_span=MIN_AXIS_SPAN_X,
    )
    y_lim = choose_axis_limit(
        values=np.asarray(y_values, dtype=float),
        user_limit=Y_LIM,
        auto_lower_at_zero=AUTO_Y_LOWER_AT_ZERO,
        min_span=MIN_AXIS_SPAN_Y,
    )
    return x_lim, y_lim


def plot_control_mixing_vs_perturb_pearson(
    df_plot: pd.DataFrame,
    sub_path: Path,
    df_s0: Optional[pd.Series] = None,
    save_dir: Optional[Path] = None,
    figsize: Tuple[float, float] = FIGSIZE,
    global_x_lim: Optional[Tuple[float, float]] = None,
    global_y_lim: Optional[Tuple[float, float]] = None,
):
    df_plot = df_plot.copy().reset_index(drop=True)

    mean_x = float(np.nanmean(df_plot[X_COLUMN].to_numpy(dtype=float)))
    mean_y = float(np.nanmean(df_plot[Y_COLUMN].to_numpy(dtype=float)))

    user_x_lim = get_user_axis_limit(X_LIM, X_LIM_BY_DATASET_GROUP, sub_path)
    user_y_lim = get_user_axis_limit(Y_LIM, Y_LIM_BY_DATASET_GROUP, sub_path)

    if global_x_lim is not None and user_x_lim is None:
        x_lim = global_x_lim
    else:
        x_lim = choose_axis_limit(
            values=np.concatenate([
                df_plot[X_COLUMN].to_numpy(dtype=float),
                np.asarray([float(df_s0[X_COLUMN])], dtype=float) if SHOW_S0_POINT and df_s0 is not None else np.asarray([], dtype=float),
            ]),
            user_limit=user_x_lim,
            auto_lower_at_zero=AUTO_X_LOWER_AT_ZERO,
            min_span=MIN_AXIS_SPAN_X,
        )

    if global_y_lim is not None and user_y_lim is None:
        y_lim = global_y_lim
    else:
        y_lim = choose_axis_limit(
            values=np.concatenate([
                df_plot[Y_COLUMN].to_numpy(dtype=float),
                np.asarray([float(df_s0[Y_COLUMN])], dtype=float) if SHOW_S0_POINT and df_s0 is not None else np.asarray([], dtype=float),
            ]),
            user_limit=user_y_lim,
            auto_lower_at_zero=AUTO_Y_LOWER_AT_ZERO,
            min_span=MIN_AXIS_SPAN_Y,
        )

    dataset_group_title = get_dataset_group_title(sub_path)
    print(
        f"[Axis] {dataset_group_title} | X_LIM used = {x_lim} | Y_LIM used = {y_lim} | "
        f"mean_x = {mean_x:.6g} | mean_y = {mean_y:.6g}"
    )

    x_raw = df_plot[X_COLUMN].to_numpy(dtype=float)
    y_raw = df_plot[Y_COLUMN].to_numpy(dtype=float)

    if DEOVERLAP_POINTS:
        x_draw, y_draw = deoverlap_xy(
            x_values=x_raw,
            y_values=y_raw,
            x_min=x_lim[0],
            x_max=x_lim[1],
            y_min=y_lim[0],
            y_max=y_lim[1],
            min_dist_fraction=MIN_POINT_DISTANCE_FRACTION,
            n_iter=POINT_REPEL_ITERATIONS,
        )
    else:
        x_draw, y_draw = x_raw.copy(), y_raw.copy()

    df_plot["_x_draw"] = x_draw
    df_plot["_y_draw"] = y_draw

    fig, ax = plt.subplots(figsize=figsize)

    # Mean lines across selected strategies included in this panel.
    if SHOW_MEAN_LINES:
        ax.axvline(
            mean_x,
            color=MEAN_LINE_COLOR,
            linestyle=MEAN_LINESTYLE,
            linewidth=MEAN_LINEWIDTH,
            alpha=MEAN_LINE_ALPHA,
            zorder=2,
        )
        ax.axhline(
            mean_y,
            color=MEAN_LINE_COLOR,
            linestyle=MEAN_LINESTYLE,
            linewidth=MEAN_LINEWIDTH,
            alpha=MEAN_LINE_ALPHA,
            zorder=2,
        )

    if DEOVERLAP_POINTS and SHOW_DISPLACEMENT_CONNECTORS:
        norm_displacement = np.sqrt(
            ((x_draw - x_raw) / max(x_lim[1] - x_lim[0], 1e-12)) ** 2
            + ((y_draw - y_raw) / max(y_lim[1] - y_lim[0], 1e-12)) ** 2
        )
        for i in range(len(df_plot)):
            if norm_displacement[i] <= 1e-4:
                continue
            ax.plot(
                [x_raw[i], x_draw[i]],
                [y_raw[i], y_draw[i]],
                color="#777777",
                linewidth=0.6,
                alpha=0.45,
                zorder=3,
            )

    for _, row in df_plot.iterrows():
        ax.scatter(
            float(row["_x_draw"]),
            float(row["_y_draw"]),
            s=POINT_SIZE,
            color=row["plot_color"],
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=5,
            alpha=0.98,
        )

    # Retained S0 point. No S0 reference lines are drawn.
    if SHOW_S0_POINT and df_s0 is not None:
        ax.scatter(
            float(df_s0[X_COLUMN]),
            float(df_s0[Y_COLUMN]),
            s=S0_POINT_SIZE,
            marker=S0_POINT_MARKER,
            color=S0_POINT_COLOR,
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=7,
            alpha=S0_POINT_ALPHA,
        )

    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)

    ax.set_xlabel(X_LABEL, fontsize=14)
    ax.set_ylabel(Y_LABEL, fontsize=14)
    ax.set_title(PLOT_TITLE, fontsize=17, weight="bold")

    if SHOW_GRID:
        ax.grid(True, linewidth=0.5, alpha=0.23, zorder=-5)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.text(
        0.02,
        0.95,
        dataset_group_title,
        transform=ax.transAxes,
        fontsize=13.5,
        color="#555555",
        ha="left",
        va="bottom",
    )

    if SHOW_MEAN_LINES and SHOW_MEAN_LINE_LABELS_ON_PLOT:
        ax.text(
            0.02,
            0.02,
            f"Mean selected x = {mean_x:.3f}\nMean selected y = {mean_y:.3f}",
            transform=ax.transAxes,
            fontsize=MEAN_LABEL_FONT_SIZE,
            color="#555555",
            ha="left",
            va="bottom",
        )

    ax.text(
        0.98,
        0.04,
        "Better perturbation recovery ↑\nHigher control mixing →",
        transform=ax.transAxes,
        fontsize=10.5,
        color="#555555",
        ha="right",
        va="bottom",
    )

    make_bottom_legend(ax, df_plot=df_plot, mean_x=mean_x, mean_y=mean_y, df_s0=df_s0)

    fig.subplots_adjust(bottom=BOTTOM_MARGIN)

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        if SAVE_PNG:
            png_path = save_dir / f"{SAVE_NAME}.png"
            fig.savefig(png_path, dpi=300, bbox_inches="tight")
            print(f"[Saved] {png_path}")
        if SAVE_SVG:
            svg_path = save_dir / f"{SAVE_NAME}.svg"
            fig.savefig(svg_path, bbox_inches="tight")
            print(f"[Saved] {svg_path}")

    if DISPLAY_FIGURES and display is not None:
        display(fig)

    plt.close(fig)
    return fig, ax


def run_all(root_dir: Path) -> None:
    plot_items = collect_plot_inputs(root_dir)
    if not plot_items:
        raise FileNotFoundError("No valid selected-variant metric tables were found.")

    global_x_lim = None
    global_y_lim = None
    if USE_GLOBAL_AUTO_LIMITS:
        global_x_lim, global_y_lim = compute_global_axis_limits(plot_items)
        print(f"[Global axis limits] X_LIM = {global_x_lim} | Y_LIM = {global_y_lim}")

    for item in plot_items:
        sub_path = item["sub_path"]
        df_plot = item["df_plot"]
        df_s0 = item.get("df_s0", None)
        save_dir = relative_save_dir(root_dir, sub_path)
        plot_control_mixing_vs_perturb_pearson(
            df_plot=df_plot,
            sub_path=sub_path,
            df_s0=df_s0,
            save_dir=save_dir,
            global_x_lim=global_x_lim,
            global_y_lim=global_y_lim,
        )


if __name__ == "__main__":
    ROOT_DIR = Path("/ibex/user/chenj0i/Perturbation/evaluation")
    run_all(ROOT_DIR)


In [ ]:
# ============================================================
# Optional manual axis limits
# ============================================================
# Default:
#   X_LIM = None
#   Y_LIM = None
#
# Suggested fixed scale if both metrics are bounded around 0-1:
#   X_LIM = (0.0, 1.0)
#   Y_LIM = (0.0, 1.0)
#
# If perturbation-effect Pearson can be negative, use:
#   X_LIM = (-0.2, 1.0)
#   Y_LIM = (0.0, 1.0)

X_LIM = None
Y_LIM = None

X_LIM_BY_DATASET_GROUP = {
    # ("Replogle_K562_essential", "single"): (0.0, 1.0),
    # ("NormanWeissman2019", "dual"): (-0.2, 1.0),
}

Y_LIM_BY_DATASET_GROUP = {
    # ("Replogle_K562_essential", "single"): (0.0, 1.0),
    # ("NormanWeissman2019", "dual"): (0.0, 1.0),
}